# RSNA Knee — Baseline v1 (CV protocol + submission)

**Purpose.** Produce the *first real baseline numbers* for this competition under the rules in
`AGENTS.md` / `rules.md`: a macro-AUC on the 58 gold studies with an honest variance estimate,
and a valid `submission.csv`. Nothing here is tuned. This notebook is the yardstick every later
experiment is measured against.

**What it does differently from the public starter**

| | starter | this notebook |
|---|---|---|
| Validation | one number on all 58 gold studies | repeated stratified folds × multiple training seeds → **mean ± σ** |
| Model selection | best epoch chosen **on gold** (fits the validation set) | **fixed epoch count**, gold is never used to select anything |
| Undefined AUC | silently skipped | `(fold, label)` cells **counted and reported** — two runs with different counts are not comparable |
| Internet | assumes `pretrained=True` downloads | offline-safe: falls back loudly, weights may come from an attached dataset |
| Submission | written at the end | written **first** as 0.5, then overwritten — a crash still leaves a valid file |
| Efficiency | not measured | per-study inference seconds + 9 h extrapolation (Efficiency Prize is separately scored) |

**Outputs** (all under `/kaggle/working`): `submission.csv`, `baseline_v1_results.json`,
`gold_probs_seed<N>.npy`, `knee_baseline_seed<N>.pt`.

**Before running:** `RUN_MODE` in the config cell decides the cost. `smoke` ≈ minutes,
`full` is the one that produces the baseline. Per `AGENTS.md` §0 the `full` run needs an approved
plan — see `plans/baseline-v1.md`.

In [ ]:
# ================================================================== CONFIG — the only cell to edit
from __future__ import annotations
import glob, json, os, re, time, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd

RUN_MODE = "full"          # "smoke" (minutes, proves it runs) | "full" (the baseline) | "submit" (inference only)

# ---- competition constants (rules.md) -----------------------------------------------------------
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_LABEL = len(TARGETS)
SUBMISSION_NAME = "submission.csv"        # rules.md: hard requirement
KAGGLE_LIMIT_H  = 9.0                     # rules.md: CPU or GPU notebook <= 9 h
WORKING_LIMIT_H = 6.75                    # rules.md: 9 h minus 25% headroom

# ---- versioned artefacts (rules.md: cache is keyed by preprocessing version) --------------------
LABELLER_VERSION  = "v1-keyword"          # weak-label rules; bump when the labeller changes
PREPROC_VERSION   = "p1"                  # bump on ANY change to the DICOM -> tensor path
EXPERIMENT_ID     = "baseline-v1"

# ---- study -> tensor geometry -------------------------------------------------------------------
SLOTS       = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1)]   # (plane, prefer fluid-sensitive)
N_SLOT      = len(SLOTS)
N_TRIPLET   = 4            # windows per slot; a window = 3 adjacent slices stacked as channels
IMG         = 192
CROP_MM     = 130.0
SLICE_BAND  = (0.15, 0.85)
K           = N_SLOT * N_TRIPLET

# ---- model / training ---------------------------------------------------------------------------
BACKBONE     = "resnet18"
PRETRAINED   = True        # with internet off this needs an attached weights dataset; see below
EPOCHS       = 4           # FIXED. Never chosen by looking at gold (rules.md hard rule 2)
BATCH        = 8
LR_HEAD      = 3e-4
LR_BACKBONE  = 1e-4
NUM_WORKERS  = 2

# ---- evaluation protocol (rules.md: statistical rules) ------------------------------------------
N_SEEDS   = {"smoke": 1, "full": 3, "submit": 1}[RUN_MODE]   # training seeds -> seed variance
SEEDS     = [2026, 2027, 2028][:N_SEEDS]
N_FOLDS   = 5              # evaluation folds over the 58 gold studies (NOT training folds)
EVAL_REPEATS = 5           # fold reshuffles; sigma comes from (seed x repeat x fold) cells
# Decoding DICOM is the real cost (CPU, ~1-4 s/study cold). 1200 studies keeps the FIRST run inside
# one session; raise it once the cache is built and saved as a Kaggle dataset — see plans/baseline-v1.md.
MAX_TRAIN_STUDIES = {"smoke": 120, "full": 1200, "submit": 0}[RUN_MODE]
CACHE_INPUT_DIRS  = ["/kaggle/input/rsna-knee-cache-p1"]   # prebuilt tensor cache, if attached

# ---- offline weights (rules.md: internet is disabled in the rerun) ------------------------------
# Backbone weights ship as an attached dataset because the rerun cannot download anything.
# Dataset: kaggle.com/datasets/vaibhav486/timm-backbones-offline (Apache-2.0, redistributable —
# required by the winners' obligation to publish weights).
TIMM_OFFLINE_DIRS = ["/kaggle/input/timm-backbones-offline", "/kaggle/input/timm-weights"]
BACKBONE_WEIGHTS = {                      # timm model name -> file in the dataset above
    "resnet18":           "resnet18.a1_in1k.bin",
    "resnet34":           "resnet34.a1_in1k.bin",
    "tf_efficientnet_b0": "tf_efficientnet_b0.ns_jft_in1k.bin",
    "convnext_tiny":      "convnext_tiny.in12k_ft_in1k.bin",
}
CHECKPOINT_DIRS   = ["/kaggle/input/rsna-knee-baseline-v1"]   # our own trained weights, for "submit"

# ---- paths ---------------------------------------------------------------------------------------
def find_root() -> Path:
    for c in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"), Path(".")]:
        if (c / "train.csv").exists() or list(c.glob("train*.csv")):
            return c
    raise FileNotFoundError("Competition data not found; set ROOT by hand.")

def find_csv(root: Path, stem: str) -> Path:
    exact = root / f"{stem}.csv"
    if exact.exists():
        return exact
    hits = sorted(c for c in root.glob(f"{stem}*.csv") if "_series" not in c.name)
    if not hits:
        raise FileNotFoundError(f"{stem}.csv not found under {root}")
    return hits[0]

ROOT  = find_root()
WORK  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / f"cache_{PREPROC_VERSION}"          # version in the path: a stale cache cannot be reused
CACHE.mkdir(parents=True, exist_ok=True)
# A prebuilt cache attached as a read-only dataset is searched first, then the writable one.
CACHE_READ = [Path(d) for d in CACHE_INPUT_DIRS if Path(d).exists()] + [CACHE]

T_START = time.time()
def elapsed_h() -> float:
    return (time.time() - T_START) / 3600.0

def budget_check(stage: str) -> None:
    """rules.md hard rule 7: stay inside the 6.75 h working limit, loudly."""
    h = elapsed_h()
    print(f"[budget] {stage}: {h:.2f} h of {WORKING_LIMIT_H} h used ({h / KAGGLE_LIMIT_H:.0%} of the Kaggle cap)")
    if h > WORKING_LIMIT_H:
        warnings.warn(f"OVER THE WORKING BUDGET at '{stage}' — this configuration is not submittable.")

np.random.seed(SEEDS[0])
print(f"run mode   : {RUN_MODE}   seeds={SEEDS}   epochs={EPOCHS}")
print(f"data root  : {ROOT.resolve()}")
print(f"work dir   : {WORK.resolve()}")
print(f"cache      : {CACHE.name}  (preproc {PREPROC_VERSION}, labeller {LABELLER_VERSION})")
print(f"per study  : {K} windows ({N_SLOT} slots x {N_TRIPLET} triplets) at {IMG}x{IMG}")

## 0. Write a valid submission *before* anything can fail

`rules.md` hard rule 10: `submission.csv` is written unconditionally, including on the failure
path. A 0.5-everywhere file scores ~0.5 AUC — worthless, but it is a *scored* submission rather
than a failed one, and it is the benchmark the Efficiency Prize measures against.

In [ ]:
test = pd.read_csv(find_csv(ROOT, "test"))
test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)

def write_submission(frame: pd.DataFrame) -> Path:
    """Single place that writes the file, so the contract is enforced in one spot."""
    out = frame.copy()
    out["StudyInstanceUID"] = out["StudyInstanceUID"].astype(str)
    out = out[["StudyInstanceUID"] + TARGETS]                 # exact column order
    out[TARGETS] = out[TARGETS].astype(float).clip(0.0, 1.0)  # exact range
    out[TARGETS] = out[TARGETS].fillna(0.5)                   # never a NaN
    path = WORK / SUBMISSION_NAME
    out.to_csv(path, index=False)
    return path

fallback = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"]})
for c in TARGETS:
    fallback[c] = 0.5
print("fallback submission written to:", write_submission(fallback))
print(f"test rows visible here: {len(test)}  (the hidden test set is substituted at rerun)")

## 1. Data, and the gold / weak split

58 studies carry human labels. 4,349 carry only a free-text report. That ratio is the whole
problem: the gold set is too small to train on and barely large enough to measure with — which is
why section 3 spends so much effort on how the measurement is done.

In [ ]:
train        = pd.read_csv(find_csv(ROOT, "train"))
train_series = pd.read_csv(find_csv(ROOT, "train_series"))
test_series  = pd.read_csv(find_csv(ROOT, "test_series"))
for df in (train, test, train_series, test_series):
    for col in ("StudyInstanceUID", "SeriesInstanceUID"):
        if col in df.columns:
            df[col] = df[col].astype(str)

gold_mask = train[TARGETS].notna().all(axis=1)
gold = train.loc[gold_mask].reset_index(drop=True)
Y_GOLD = gold[TARGETS].values.astype(int)          # [58, 12] — the only ground truth we own

pos = pd.Series(Y_GOLD.sum(0), index=TARGETS)
print(f"gold studies: {len(gold)}   report-only: {(~gold_mask).sum()}")
print("\npositives per target among the gold studies:")
print(pos.to_string())
print(f"\nrarest target: {pos.idxmin()} with {pos.min()} positives -> "
      f"{pos.min() / N_FOLDS:.1f} expected positives per fold. "
      "This is why undefined (fold, label) cells are unavoidable and must be counted.")

## 2. Weak labels from the reports — `v1-keyword`

Carried over from the public starter and **frozen** as version `v1-keyword` so later labeller work
has something to beat (`rules.md`: every experiment states its labeller version and that version's
measured macro AUC against gold). Negation handling is part of the contract and is unit-tested
below — "sin rotura" must never become a 1.

In [ ]:
def normalise(text: str) -> str:
    t = unicodedata.normalize("NFKD", str(text))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", t.lower())

NEG = (r"(?:no |not |without |absence of |negative for |intact |normal |unremarkable |ohne |"
       r"kein[e]?[nrms]? |unauff|sin |ausencia|geen |zonder |normale |normaal |bez |uredn|"
       r"nema |sans |pas de |absence)")

PATTERNS = {
    "ACL": r"(acl|anterior cruciate|lca|vkb|ligamento cruzado anterior|voorste kruisband|"
           r"kruisband anterior|prednj[ei] krizn|kreuzband(?:ruptur)?\s*(?:vorder)?|vorderes kreuzband)",
    "MCL": r"(mcl|medial collateral|ligamento colateral medial|innenband|mediale[nr]? kollateralband|"
           r"mediale collaterale|medijalni kolateralni)",
    "Medial Meniscus": r"(medial meniscus|menisco (?:interno|medial)|innenmeniskus|mediale meniscus|"
                       r"medijalni menisk|meniscus medialis|meniscus internus)",
    "Lateral Meniscus": r"(lateral meniscus|menisco (?:externo|lateral)|aussenmeniskus|laterale meniscus|"
                        r"lateralni menisk|meniscus lateralis)",
    "Medial OA": None, "Lateral OA": None, "PF OA": None,       # compartment co-occurrence, below
    "Effusion": r"(effusion|derrame|gelenkerguss|ergus[s]?|hydrops|izljev|epanchement|joint fluid|"
                r"gewrichtsvocht|vocht)",
    "Synovitis": r"(synovit\w*|sinovit\w*|synovialit\w*|sinovij\w*|"
                 r"synovial\w* (?:proliferation|thickening|verdikking|hypertroph\w*)|"
                 r"proliferacij\w* sinovij|pannus|synoviale? reizung)",
    "Baker's": r"(baker|popliteal cyst|quiste de baker|bakerzyste|baker-zyste|bakerova cist|kyste de baker)",
    "Contusion": r"(bone (?:marrow )?(?:contusion|bruise|oedema|edema)|contusion|knochenmarkod|kontuzij|"
                 r"botcontusie|edema oseo)",
    "Fracture": r"(fracture|fractur|fraktur|fisura osea|prijelom|breuk|fractuur|avulsion)",
}
ABNORMAL = (r"(tear|rupt|riss|scheur|rotura|lesion|desgarr|lasion|laesion|degenerativ|signal|"
            r"tearing|ruptura|insuffizienz|discontinu|abnormal)")
NEEDS_ABNORMAL = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_WORD = (r"(osteoarthrit\w*|arthros\w*|artros\w*|artroz\w*|gonarthros\w*|osteoartr\w*|chondral loss|"
           r"cartilage loss|knorpel\w*|chondropath\w*|kraakbeen\w*|hrskavic\w*|osteophyt\w*|osteofit\w*|"
           r"degenerative (?:change|veranderung)\w*|denudacij\w*)")
COMPARTMENT = {
    "Medial OA":  r"(medial\w*|mediaal|medijaln\w*|intern[oa]|innen\w*|inner)",
    "Lateral OA": r"(lateral\w*|lateraal|lateraln\w*|extern[oa]|aussen\w*|outer)",
    "PF OA":      r"(patell\w*|patel\w*|femoropatel\w*|retropatell\w*|trochlea\w*|trohlej\w*)",
}
WINDOW = 60

def _oa_label(t: str, key: str) -> float:
    pos = neg = 0
    for m in re.finditer(OA_WORD, t):
        s, e = m.span()
        ctx = t[max(0, s - WINDOW):e + WINDOW]
        if not re.search(COMPARTMENT[key], ctx):
            continue
        if key != "PF OA" and re.search(r"(?:patell|patel|trochlea|trohlej)", ctx):
            continue
        if re.search(NEG + r"[^.]{0,25}$", t[max(0, s - 45):s]):
            neg += 1
        else:
            pos += 1
    return 1.0 if pos else (0.0 if neg else np.nan)

def label_report(text: str) -> dict:
    """{finding: 1.0 | 0.0 | nan}. nan = the report does not say — NOT a zero (rules.md)."""
    t = normalise(text)
    out = {k: _oa_label(t, k) for k in COMPARTMENT}
    for key, pattern in PATTERNS.items():
        if pattern is None:
            continue
        pos = neg = 0
        for m in re.finditer(pattern, t):
            s, e = m.span()
            left, right = t[max(0, s - 45):s], t[e:e + 80]
            negated = (re.search(NEG + r"[^.]{0,25}$", left)
                       or re.search(r"^\W{0,4}(?:" + NEG + r"|ist intakt|intacto|intact)", right))
            if key in NEEDS_ABNORMAL:
                near_abnormal = re.search(ABNORMAL, right[:60]) or re.search(ABNORMAL + r"[^.]{0,30}$", left)
                if not near_abnormal:
                    if re.search(r"^\W{0,6}(?:" + NEG + r"|intact|normal)", right):
                        neg += 1
                    continue
            if negated:
                neg += 1
            else:
                pos += 1
        out[key] = 1.0 if pos else (0.0 if neg else np.nan)
    return out

weak_labels = pd.DataFrame([label_report(r) for r in train["Report"]])[TARGETS]
weak_labels.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values)
print("label coverage (share of studies the report can decide):")
print((weak_labels[TARGETS].notna().mean() * 100).round(1).to_string())

In [ ]:
# ---- unit tests: the labeller contract (rules.md — negation tests ship with every labeller) ----
def _t(text, key):
    return label_report(text)[key]

def test_labeller_contract():
    assert _t("Rotura del ligamento cruzado anterior.", "ACL") == 1.0,        "positive ACL tear missed"
    assert _t("Sin rotura del ligamento cruzado anterior.", "ACL") == 0.0,    "'sin rotura' read as positive"
    assert _t("ACL intact.", "ACL") == 0.0,                                   "'intact' not read as negative"
    assert _t("Vorderes Kreuzband rupturiert.", "ACL") == 1.0,                "German positive missed"
    assert _t("Rotura de menisco interno.", "Medial Meniscus") == 1.0,        "medial meniscus tear missed"
    assert np.isnan(_t("Rotura de menisco interno.", "Lateral Meniscus")),    "medial tear leaked to lateral"
    assert _t("Derrame articular.", "Effusion") == 1.0,                       "effusion missed"
    assert np.isnan(_t("Estudio de rodilla.", "Fracture")),                   "silence must be unknown, not 0"
    assert _t("Artrosis femorotibial medial.", "Medial OA") == 1.0,           "medial OA missed"
    assert np.isnan(_t("Artrosis femorotibial medial.", "Lateral OA")),       "medial OA leaked to lateral"
    assert _t("Artrosis patelofemoral.", "PF OA") == 1.0,                     "PF OA missed"
    print("labeller contract: 11/11 assertions pass")


def test_known_gaps_of_v1():
    """Gaps found while writing these tests. They assert what v1 *currently does*, so that any
    future labeller version trips them — which is the signal to bump LABELLER_VERSION and re-measure.
    Spanish is the dominant report language, so these are not exotic misses."""
    assert np.isnan(_t("Condropatia retropatelar.", "PF OA")),  "v1 gap closed: bump LABELLER_VERSION"
    assert np.isnan(_t("Condropatia rotuliana grado II.", "PF OA")), "v1 gap closed: bump LABELLER_VERSION"
    assert np.isnan(_t("Perdida de cartilago patelar.", "PF OA")),   "v1 gap closed: bump LABELLER_VERSION"
    print("known v1 gaps (Spanish 'condropatia' / 'cartilago' unmatched): still present, as expected")

test_labeller_contract()
test_known_gaps_of_v1()

In [ ]:
from sklearn.metrics import roc_auc_score

# The labeller's own macro AUC on gold, with unknowns scored 0.5 (an uninformative guess).
# This is the number every future labeller version must beat, and it is the floor for the
# image model: a model trained on these labels inherits their errors.
gold_weak = weak_labels.loc[gold_mask.values, TARGETS].reset_index(drop=True)
rows, aucs = [], []
for j, name in enumerate(TARGETS):
    p = gold_weak[name].fillna(0.5).values
    auc = roc_auc_score(Y_GOLD[:, j], p) if len(np.unique(Y_GOLD[:, j])) > 1 else np.nan
    aucs.append(auc)
    rows.append((name, int(Y_GOLD[:, j].sum()), round(gold_weak[name].notna().mean(), 3), auc))
LABELLER_AUC = float(np.nanmean(aucs))
print(pd.DataFrame(rows, columns=["target", "gold pos", "coverage", "AUC"]).round(3).to_string(index=False))
print(f"\n{LABELLER_VERSION} macro AUC on gold: {LABELLER_AUC:.4f}   <- record this in experiments.md")

## 3. The evaluation protocol

This is the part that makes every later `+1`/`-1` verdict meaningful, so it is spelled out.

**The model never trains on gold.** It trains on weak labels and is *measured* on the 58 gold
studies. So "CV" here does not mean a train/validation split — it means a **variance estimate** of
a measurement taken on 58 samples. Two sources of variance are separated:

- **evaluation variance** — resample the 58 studies into `N_FOLDS` stratified folds, `EVAL_REPEATS`
  times, and score each fold. This says how much the number moves because of *which patients* we
  happen to have.
- **seed variance** — retrain with `N_SEEDS` different seeds. This says how much it moves because
  of *training randomness*.

A future experiment is compared **paired**: same folds, same repeats, same seeds, cell by cell.
σ for the `Δ > 2σ` test in `rules.md` comes from the spread of those paired cells.

**Undefined cells.** With 9 positives in the rarest target, some (fold, label) pairs contain no
positive and their AUC does not exist. They are dropped and the surviving cells averaged — never
imputed at 0.5, which would silently pull every score toward chance. The dropped count is reported
with every score, because two runs that dropped different cells are not comparable.

In [ ]:
def multilabel_folds(y: np.ndarray, n_folds: int, seed: int) -> np.ndarray:
    """Greedy multi-label stratification: rarest target first, each study to the fold that is
    furthest below its quota. Deterministic given `seed`. Returns fold index per study."""
    n, m = y.shape
    rng = np.random.default_rng(seed)
    order_targets = np.argsort(y.sum(0))                      # rarest target first
    fold_of = np.full(n, -1, int)
    counts = np.zeros((n_folds, m), float)
    fold_size = np.zeros(n_folds, int)
    quota = n / n_folds

    for j in order_targets:
        idx = np.where((y[:, j] == 1) & (fold_of < 0))[0]
        rng.shuffle(idx)
        for i in idx:
            deficit = counts[:, j].min() - counts[:, j]        # most negative = most over quota
            best = np.lexsort((rng.random(n_folds), fold_size - quota, -deficit))[0]
            fold_of[i] = best
            counts[best] += y[i]
            fold_size[best] += 1

    rest = np.where(fold_of < 0)[0]                            # all-negative studies
    rng.shuffle(rest)
    for i in rest:
        best = np.lexsort((rng.random(n_folds), fold_size))[0]
        fold_of[i] = best
        counts[best] += y[i]
        fold_size[best] += 1
    return fold_of


def fold_cells(y: np.ndarray, p: np.ndarray, fold_of: np.ndarray, n_folds: int):
    """-> (per-fold macro AUC, n_dropped, n_cells). A (fold, label) cell with no positive or no
    negative has no AUC: it is dropped, never imputed."""
    per_fold, dropped, total = [], 0, 0
    for f in range(n_folds):
        sel = fold_of == f
        aucs = []
        for j in range(y.shape[1]):
            total += 1
            yt = y[sel, j]
            if len(np.unique(yt)) < 2:
                dropped += 1
                continue
            aucs.append(roc_auc_score(yt, p[sel, j]))
        per_fold.append(float(np.mean(aucs)) if aucs else np.nan)
    return per_fold, dropped, total


def full_macro_auc(y: np.ndarray, p: np.ndarray) -> float:
    """Macro AUC on all 58 at once — the estimate most comparable to the leaderboard."""
    aucs = [roc_auc_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if len(np.unique(y[:, j])) > 1]
    return float(np.mean(aucs))


def evaluate(y: np.ndarray, p: np.ndarray, n_folds=N_FOLDS, repeats=EVAL_REPEATS, tag="") -> dict:
    """The one function every experiment must use to report a score."""
    cells, dropped, total = [], 0, 0
    for r in range(repeats):
        fold_of = multilabel_folds(y, n_folds, seed=1000 + r)
        vals, d, t = fold_cells(y, p, fold_of, n_folds)
        cells.extend(vals); dropped += d; total += t
    cells = np.array(cells, float)
    return {"tag": tag, "full_macro_auc": full_macro_auc(y, p),
            "fold_mean": float(np.nanmean(cells)), "fold_std": float(np.nanstd(cells)),
            "cells": cells.tolist(), "dropped_cells": dropped, "total_cells": total,
            "dropped_pct": round(100 * dropped / max(total, 1), 1)}

In [ ]:
# ---- unit tests: the evaluation protocol itself ------------------------------------------------
def test_protocol():
    rng = np.random.default_rng(0)
    y = (rng.random((58, N_LABEL)) < 0.25).astype(int)

    f = multilabel_folds(y, N_FOLDS, seed=0)
    sizes = np.bincount(f, minlength=N_FOLDS)
    assert set(np.unique(f)) == set(range(N_FOLDS)), "some fold is empty"
    assert sizes.max() - sizes.min() <= 2, f"folds badly unbalanced: {sizes}"
    assert (multilabel_folds(y, N_FOLDS, seed=0) == f).all(), "folds are not deterministic"
    assert not (multilabel_folds(y, N_FOLDS, seed=1) == f).all(), "different seeds gave identical folds"

    perfect = y.astype(float) * 0.9 + 0.05
    assert abs(evaluate(y, perfect)["full_macro_auc"] - 1.0) < 1e-9, "perfect predictions must score 1.0"
    chance = evaluate(y, np.full_like(perfect, 0.5))["full_macro_auc"]
    assert abs(chance - 0.5) < 1e-9, f"constant predictions must score 0.5, got {chance}"

    # a target with no positives must be DROPPED, not scored as 0.5
    y2 = y.copy(); y2[:, 0] = 0
    r = evaluate(y2, perfect)
    assert r["dropped_cells"] >= N_FOLDS * EVAL_REPEATS, "all-negative target was not dropped"
    assert r["full_macro_auc"] > 0.99, "dropping leaked chance-level cells into the mean"
    print(f"evaluation protocol: all assertions pass "
          f"(fold sizes {sizes.tolist()}, dropped {r['dropped_pct']}% in the degenerate case)")

test_protocol()
budget_check("after tests")

## 4. DICOM → tensor

Unchanged from the starter in substance — one series per plane, four windows per series, three
adjacent slices as channels, a physical-millimetre centre crop so knees of different sizes land at
the same scale. Two rule-driven changes: the cache directory carries `PREPROC_VERSION`, so a
preprocessing change cannot silently reuse stale tensors, and decode failures degrade to a masked
window instead of raising.

In [ ]:
import pydicom
import cv2
cv2.setNumThreads(1)                       # parallelism is across studies, not inside OpenCV

SERIES_DIR_TRAIN = ROOT / "train_series"
SERIES_DIR_TEST  = ROOT / "test_series"
HAVE_IMAGES = SERIES_DIR_TRAIN.exists() or SERIES_DIR_TEST.exists()
print("image directories present:", HAVE_IMAGES)
if not HAVE_IMAGES:
    print("  -> metadata-only environment: training is skipped, the fallback submission stands.")

series_by_study      = {k: v.to_dict("records") for k, v in train_series.groupby("StudyInstanceUID")}
series_by_study_test = {k: v.to_dict("records") for k, v in test_series.groupby("StudyInstanceUID")}

def pick_series(rows, plane, fluid, used):
    cand = [r for r in rows if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]
    pref = [r for r in cand if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
    return (pref or cand or [None])[0]

def ordered_slices(series_dir: Path):
    keyed = []
    for f in glob.glob(str(series_dir / "*.dcm")):
        try:
            hdr = pydicom.dcmread(f, stop_before_pixels=True)
            pos = int(getattr(hdr, "InstanceNumber", 0) or 0)
            spacing = float(hdr.PixelSpacing[0]) if hasattr(hdr, "PixelSpacing") else 0.0
        except Exception:
            continue
        keyed.append((pos, f, spacing))
    keyed.sort()
    return [(f, s) for _, f, s in keyed]

def read_pixels(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = arr.max() - arr
        return arr
    except Exception:
        return None

def crop_and_resize(arr, spacing):
    h, w = arr.shape
    if spacing <= 0:
        spacing = CROP_MM / max(h, w)
    side = min(int(round(CROP_MM / spacing)), h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    return cv2.resize(arr[y0:y0 + side, x0:x0 + side], (IMG, IMG), interpolation=cv2.INTER_AREA)

def build_study(study_uid: str, rows, series_dir: Path):
    """-> (uint8 [K,3,IMG,IMG], bool [K]). A missing or unreadable slot is zeros with mask False."""
    volume = np.zeros((K, 3, IMG, IMG), np.uint8)
    mask = np.zeros(K, bool)
    used, w = set(), 0
    for plane, fluid in SLOTS:
        record = pick_series(rows, plane, fluid, used)
        if record is None:
            w += N_TRIPLET; continue
        used.add(record["SeriesInstanceUID"])
        files = ordered_slices(series_dir / study_uid / record["SeriesInstanceUID"])
        n = len(files)
        if n == 0:
            w += N_TRIPLET; continue
        lo, hi = int(n * SLICE_BAND[0]), max(int(n * SLICE_BAND[1]) - 1, 0)
        centres = np.linspace(lo, max(hi, lo), N_TRIPLET).round().astype(int)
        med = float(np.median([s for _, s in files if s > 0]) if any(s > 0 for _, s in files) else 0.0)
        for centre in centres:
            idx = [int(np.clip(centre + d, 0, n - 1)) for d in (-1, 0, 1)]
            planes, spacings = [], []
            for i in idx:
                path, spacing = files[i]
                planes.append(read_pixels(path))
                spacings.append(spacing if spacing > 0 else med)
            present = [p for p in planes if p is not None]
            if not present:
                w += 1; continue
            low, high = np.percentile(np.concatenate([p.ravel() for p in present]), [2.0, 98.0])
            for c, (p, spacing) in enumerate(zip(planes, spacings)):
                if p is None:
                    continue
                norm = np.clip((p - low) / (high - low + 1e-6), 0, 1)
                volume[w, c] = (crop_and_resize(norm, spacing) * 255).astype(np.uint8)
            mask[w] = True
            w += 1
    return volume, mask

def cached_study(study_uid: str, rows, series_dir: Path):
    """DICOM decoding, not the GPU, is the bottleneck — build once per (study, PREPROC_VERSION)."""
    for d in CACHE_READ:                          # attached prebuilt cache first, then our own
        path = d / f"{study_uid}.npz"
        if path.exists():
            try:
                with np.load(path) as z:
                    return z["volume"], z["mask"]
            except Exception:
                if d == CACHE:
                    path.unlink(missing_ok=True)  # corrupt entry we own: rebuild rather than crash
    path = CACHE / f"{study_uid}.npz"
    volume, mask = build_study(study_uid, rows, series_dir)
    np.savez_compressed(path, volume=volume, mask=mask)
    return volume, mask

## 5. Model

`K` windows per study go through a shared backbone; a per-target attention head decides which
windows matter for which finding (a Baker's cyst and a PF cartilage defect are not visible in the
same slice), then each target reads out its own weighted feature vector. The loss is masked BCE —
cells the report could not decide contribute nothing, they are never imputed as 0.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader

def pick_device() -> tuple[str, str]:
    """Kaggle hands out P100s (sm_60) that the installed PyTorch no longer supports — the failure
    is a `no kernel image is available` CUDA error thrown deep inside the first conv, minutes into
    a run. Detect the mismatch up front and degrade to CPU instead of crashing."""
    if not torch.cuda.is_available():
        return "cpu", "no CUDA device"
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    cap = major * 10 + minor
    supported = sorted(int(a.split("_")[1]) for a in torch.cuda.get_arch_list() if a.startswith("sm_"))
    if supported and cap < min(supported):
        warnings.warn(
            f"{name} is sm_{cap}; this PyTorch supports sm_{supported}. FALLING BACK TO CPU. "
            "Switch the notebook accelerator to 'GPU T4 x2' in the Kaggle UI (Settings -> "
            "Accelerator) — the CLI cannot select the GPU type — then re-run.")
        return "cpu", f"{name} (sm_{cap}) unsupported by this torch build"
    return "cuda", name

DEVICE, DEVICE_NOTE = pick_device()
USE_AMP = DEVICE == "cuda"
try:
    from torch.amp import GradScaler as _GS, autocast as _AC
    def make_scaler(): return _GS("cuda", enabled=USE_AMP)
    def amp_autocast(): return _AC("cuda", enabled=USE_AMP)
except ImportError:
    def make_scaler(): return torch.cuda.amp.GradScaler(enabled=USE_AMP)
    def amp_autocast(): return torch.cuda.amp.autocast(enabled=USE_AMP)

print("device:", DEVICE, "|", DEVICE_NOTE)
if DEVICE == "cpu" and torch.cuda.is_available():
    print("  !! a GPU is attached but unusable — this run will be slow and is NOT the baseline")

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def set_all_seeds(seed: int) -> None:
    """rules.md hard rule 5: same notebook, same input, same number."""
    np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def find_offline_weights() -> Path | None:
    """The attached weights file for BACKBONE, or None if no dataset is attached."""
    fname = BACKBONE_WEIGHTS.get(BACKBONE)
    if not fname:
        return None
    for d in TIMM_OFFLINE_DIRS:
        f = Path(d) / fname
        if f.exists():
            return f
    return None


def make_backbone():
    """Offline first: the rerun has no internet, so weights come from an attached dataset.
    The load is *verified* — a silently-partial load would look pretrained and train like noise."""
    net = timm.create_model(BACKBONE, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")
    if not PRETRAINED:
        return net, False

    f = find_offline_weights()
    if f is not None:
        sd = torch.load(f, map_location="cpu", weights_only=True)
        missing, unexpected = net.load_state_dict(sd, strict=False)
        # num_classes=0 drops the classifier, so those keys are expected to be unexpected.
        stray = [k for k in unexpected if not re.match(r"^(fc|classifier|head)\.", k)]
        if missing or stray:
            warnings.warn(f"offline weights loaded PARTIALLY from {f.name}: "
                          f"{len(missing)} missing, stray unexpected {stray[:5]} — treat as random init.")
            return net, False
        print(f"backbone weights: {f} (offline, {len(sd)} tensors, 0 missing)")
        return net, True

    try:                                   # training notebooks may have internet; submissions never do
        net = timm.create_model(BACKBONE, pretrained=True, num_classes=0, in_chans=3, global_pool="avg")
        print("backbone weights: downloaded from the hub (internet is ON — not the submission path)")
        return net, True
    except Exception as e:
        warnings.warn(f"NO pretrained weights ({type(e).__name__}) — RANDOM INIT. Attach "
                      f"{TIMM_OFFLINE_DIRS[0]}; a random-init score is not a baseline (plans/baseline-v1.md).")
        return net, False

class KneeStudyDataset(Dataset):
    """One item = one study: K windows, a window-validity mask, 12 possibly-unknown labels."""
    def __init__(self, uids, labels, series_map, series_dir, train_mode=False):
        self.uids, self.labels = list(uids), np.asarray(labels, np.float32)
        self.series_map, self.series_dir, self.train_mode = series_map, series_dir, train_mode
    def __len__(self):
        return len(self.uids)
    def __getitem__(self, i):
        uid = self.uids[i]
        volume, mask = cached_study(uid, self.series_map.get(uid, []), self.series_dir)
        x = torch.from_numpy(volume.astype(np.float32) / 255.0)
        if self.train_mode and np.random.rand() < 0.5:
            x = torch.flip(x, dims=[-1])
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        y = torch.from_numpy(self.labels[i])
        return x, torch.from_numpy(mask.astype(np.float32)), torch.nan_to_num(y), ~torch.isnan(y)

class KneeNet(nn.Module):
    def __init__(self, n_label=N_LABEL, drop=0.2):
        super().__init__()
        self.backbone, self.is_pretrained = make_backbone()
        dim = self.backbone.num_features
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),
                                  nn.Linear(256, n_label))
        self.cls_w = nn.Parameter(torch.zeros(n_label, dim))
        self.cls_b = nn.Parameter(torch.zeros(n_label))
        nn.init.trunc_normal_(self.cls_w, std=0.02)
    def forward(self, x, window_mask):
        b, k = x.shape[:2]
        h = self.norm(self.backbone(x.flatten(0, 1)).view(b, k, -1))
        a = self.attn(h).masked_fill(window_mask[:, :, None] < 0.5, float("-inf"))
        empty = window_mask.sum(1) == 0            # a study with no readable window: uniform, not NaN
        if empty.any():
            a[empty] = 0.0
        pooled = torch.einsum("bkn,bkf->bnf", torch.softmax(a, dim=1), h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

def masked_bce(logits, targets, target_mask):
    """Unknown label cells contribute nothing — they are masked, never filled (rules.md)."""
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none") * target_mask.float()
    return loss.sum() / target_mask.float().sum().clamp(min=1.0)

In [ ]:
# ---- unit tests: model and loss contracts ------------------------------------------------------
def test_model_contracts():
    torch.manual_seed(0)
    net = KneeNet().eval()
    x  = torch.randn(2, K, 3, 64, 64)
    wm = torch.ones(2, K); wm[1, 2:] = 0                      # study 1: only two valid windows
    with torch.no_grad():
        out = net(x, wm)
    assert out.shape == (2, N_LABEL), f"bad output shape {out.shape}"
    assert torch.isfinite(out).all(), "non-finite logits with a partially masked study"

    wm0 = torch.zeros(2, K)                                    # the pathological all-missing study
    with torch.no_grad():
        out0 = net(x, wm0)
    assert torch.isfinite(out0).all(), "all-masked study produced NaN — a submission killer"

    logits = torch.zeros(2, N_LABEL, requires_grad=True)
    y  = torch.ones(2, N_LABEL)
    m  = torch.zeros(2, N_LABEL, dtype=torch.bool); m[0, 0] = True
    l  = masked_bce(logits, y, m)
    assert abs(l.item() - float(np.log(2))) < 1e-5, "masked BCE is averaging over masked cells"
    l.backward()
    assert logits.grad[1].abs().sum() == 0, "gradient leaked through a masked cell"
    print("model + loss contracts: all assertions pass "
          f"(backbone pretrained={net.is_pretrained})")


def test_offline_weights_available():
    """The submission runs with internet OFF. If the weights dataset is attached, prove the
    offline path is the one actually used; if it is not, fail loudly rather than train noise."""
    f = find_offline_weights()
    if f is None:
        warnings.warn(f"No offline weights for '{BACKBONE}' in {TIMM_OFFLINE_DIRS}. "
                      "Fine for a smoke run with internet ON; NOT fine for the baseline or a submission.")
        return
    sd = torch.load(f, map_location="cpu", weights_only=True)
    probe = timm.create_model(BACKBONE, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")
    missing, unexpected = probe.load_state_dict(sd, strict=False)
    assert not missing, f"{len(missing)} weights missing from {f.name}: {missing[:5]}"
    assert all(re.match(r"^(fc|classifier|head)\.", k) for k in unexpected), \
        f"unexpected non-classifier keys: {unexpected[:5]}"
    print(f"offline weights: {f.name} loads cleanly ({len(sd)} tensors, only the classifier discarded)")

test_model_contracts()
test_offline_weights_available()

## 6. Train — `N_SEEDS` runs, fixed epochs, gold untouched

Each seed trains on the weakly-labelled studies for a **fixed** `EPOCHS` and then predicts the 58
gold studies once. No early stopping, no best-epoch selection: both would be fitting the validation
set, which `rules.md` hard rule 2 forbids and which would make the baseline optimistic in a way no
later experiment could correct for.

Gold probabilities are saved per seed. They are the artefact every future experiment is compared
against — paired, cell by cell.

In [ ]:
usable = weak_labels[TARGETS].notna().any(axis=1) & ~gold_mask.values
all_train_uids = weak_labels.loc[usable, "StudyInstanceUID"].tolist()
all_train_y    = weak_labels.loc[usable, TARGETS].values.astype(np.float32)
gold_uids      = gold["StudyInstanceUID"].tolist()

print(f"weakly-labelled studies available : {len(all_train_uids)}")
print(f"known label cells among them      : {np.isfinite(all_train_y).mean() * 100:.1f}%")
print(f"using this run                    : {min(MAX_TRAIN_STUDIES, len(all_train_uids))}")
print(f"measuring on                      : {len(gold_uids)} gold studies (never trained on)")


def train_one_seed(seed: int):
    """Train once; return gold probabilities [58, 12] and timing. Deterministic given `seed`."""
    set_all_seeds(seed)
    rng = np.random.default_rng(seed)
    uids, ys = all_train_uids, all_train_y
    if len(uids) > MAX_TRAIN_STUDIES:
        keep = rng.choice(len(uids), MAX_TRAIN_STUDIES, replace=False)
        uids = [uids[i] for i in keep]
        ys = ys[keep]

    train_dl = DataLoader(KneeStudyDataset(uids, ys, series_by_study, SERIES_DIR_TRAIN, True),
                          batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    gold_dl  = DataLoader(KneeStudyDataset(gold_uids, np.full((len(gold_uids), N_LABEL), np.nan, np.float32),
                                           series_by_study, SERIES_DIR_TRAIN),
                          batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

    model = KneeNet().to(DEVICE)
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": head, "lr": LR_HEAD}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD],
                                                total_steps=max(EPOCHS * len(train_dl), 1), pct_start=0.2)
    scaler = make_scaler()

    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        running = seen = 0
        te = time.time()
        for x, wm, y, ym in train_dl:
            x, wm, y, ym = x.to(DEVICE), wm.to(DEVICE), y.to(DEVICE), ym.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = masked_bce(model(x, wm), y, ym)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item() * x.size(0); seen += x.size(0)
        # Printed for monitoring only — NOT used to pick a checkpoint.
        print(f"  seed {seed} epoch {epoch + 1}/{EPOCHS}  loss {running / max(seen, 1):.4f}  "
              f"({time.time() - te:.0f}s)")

    model.eval()
    probs = []
    with torch.no_grad():
        for x, wm, _, _ in gold_dl:
            with amp_autocast():
                probs.append(torch.sigmoid(model(x.to(DEVICE), wm.to(DEVICE))).float().cpu().numpy())
    probs = np.concatenate(probs)
    torch.save(model.state_dict(), WORK / f"knee_{EXPERIMENT_ID}_seed{seed}.pt")
    np.save(WORK / f"gold_probs_seed{seed}.npy", probs)
    return model, probs, time.time() - t0

In [ ]:
models, seed_results, gpu_seconds = [], [], 0.0

if HAVE_IMAGES and RUN_MODE != "submit":
    for seed in SEEDS:
        print(f"\n--- seed {seed} ---")
        model, probs, secs = train_one_seed(seed)
        gpu_seconds += secs
        models.append(model)
        r = evaluate(Y_GOLD, probs, tag=f"seed{seed}")
        seed_results.append(r)
        print(f"  seed {seed}: full-set macro AUC {r['full_macro_auc']:.4f} | "
              f"fold mean {r['fold_mean']:.4f} ± {r['fold_std']:.4f} | "
              f"dropped cells {r['dropped_cells']}/{r['total_cells']} ({r['dropped_pct']}%)")
        budget_check(f"after seed {seed}")
elif RUN_MODE == "submit" and HAVE_IMAGES:
    # Inference-only run: weights come from an attached dataset, because the rerun has no internet.
    found = [p for d in CHECKPOINT_DIRS if Path(d).exists() for p in sorted(Path(d).glob("*.pt"))]
    for ckpt in found:
        net = KneeNet().to(DEVICE)
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        models.append(net.eval())
        print(f"loaded {ckpt.name}")
    if not models:
        warnings.warn("RUN_MODE='submit' but no checkpoint found in CHECKPOINT_DIRS — "
                      "the submission will be the 0.5 fallback. Attach the weights dataset.")
else:
    print("Skipping training (no images available).")

## 7. The baseline number

Three numbers, and they answer different questions:

- **full-set macro AUC** — the single estimate most comparable to the leaderboard. Report this as
  *the* baseline CV in `experiments.md`.
- **fold mean ± σ** — the honest spread. **σ is what every future `Δ > 2σ` test uses**, so a small
  improvement claimed against a large σ is noise by definition.
- **dropped cells** — how much of the metric was undefined. A later run with a different count is
  not comparable to this one.

In [ ]:
results = {"experiment_id": EXPERIMENT_ID, "run_mode": RUN_MODE, "date": time.strftime("%Y-%m-%d"),
           "labeller_version": LABELLER_VERSION, "labeller_gold_auc": round(LABELLER_AUC, 4),
           "preproc_version": PREPROC_VERSION, "backbone": BACKBONE, "epochs": EPOCHS,
           "seeds": SEEDS, "n_folds": N_FOLDS, "eval_repeats": EVAL_REPEATS,
           "train_studies": min(MAX_TRAIN_STUDIES, len(all_train_uids)),
           "train_gpu_seconds": round(gpu_seconds, 1)}

if seed_results:
    cells = np.concatenate([np.array(r["cells"], float) for r in seed_results])
    ens = np.mean([np.load(WORK / f"gold_probs_seed{s}.npy") for s in SEEDS], axis=0)
    ens_r = evaluate(Y_GOLD, ens, tag="seed-ensemble")
    results.update({
        "per_seed_full_auc": [round(r["full_macro_auc"], 4) for r in seed_results],
        "baseline_cv": round(float(np.mean([r["full_macro_auc"] for r in seed_results])), 4),
        "seed_spread": round(float(np.std([r["full_macro_auc"] for r in seed_results])), 4),
        "fold_mean": round(float(np.nanmean(cells)), 4),
        "sigma": round(float(np.nanstd(cells)), 4),
        "significance_threshold_2sigma": round(2 * float(np.nanstd(cells)), 4),
        "dropped_cells": sum(r["dropped_cells"] for r in seed_results),
        "total_cells": sum(r["total_cells"] for r in seed_results),
        "ensemble_full_auc": round(ens_r["full_macro_auc"], 4),
    })
    print(f"BASELINE CV (mean full-set macro AUC over {len(SEEDS)} seeds) : {results['baseline_cv']:.4f}")
    print(f"  spread across seeds                                  : ±{results['seed_spread']:.4f}")
    print(f"  fold mean ± sigma                                    : {results['fold_mean']:.4f} ± {results['sigma']:.4f}")
    print(f"  => a later change must move CV by more than           : {results['significance_threshold_2sigma']:.4f}  (2 sigma)")
    print(f"  undefined (fold,label) cells dropped                  : {results['dropped_cells']}/{results['total_cells']}")
    print(f"  {len(SEEDS)}-seed ensemble                                      : {results['ensemble_full_auc']:.4f}")
    print(f"  labeller {LABELLER_VERSION} alone                          : {LABELLER_AUC:.4f}")
    print(f"\n  per target (seed-ensemble):")
    per_t = {t: round(roc_auc_score(Y_GOLD[:, j], ens[:, j]), 3)
             for j, t in enumerate(TARGETS) if len(np.unique(Y_GOLD[:, j])) > 1}
    print(pd.Series(per_t).sort_values().to_string())
    results["per_target_auc"] = per_t
else:
    print("No training ran — no baseline number produced.")

## 8. Inference and submission

Per-study seconds are measured because the **Efficiency Prize scores runtime**, and because
`rules.md` hard rule 7 needs a per-study figure to extrapolate against the 9 h cap for a hidden
test set whose size is not published. Any study that fails for any reason keeps its 0.5 row: one
bad DICOM must never sink the submission.

In [ ]:
sub = fallback.copy()
infer_seconds, n_failed = 0.0, 0

if HAVE_IMAGES and models:
    nets = [m.eval() for m in models]                      # seed ensemble = mean of sigmoids
    ds = KneeStudyDataset(sub["StudyInstanceUID"].tolist(),
                          np.full((len(sub), N_LABEL), np.nan, np.float32),
                          series_by_study_test, SERIES_DIR_TEST)
    t0 = time.time()
    for i in range(len(ds)):
        try:
            x, wm, _, _ = ds[i]
            x, wm = x[None].to(DEVICE), wm[None].to(DEVICE)
            with torch.no_grad(), amp_autocast():
                p = np.mean([torch.sigmoid(net(x, wm))[0].float().cpu().numpy() for net in nets], axis=0)
            sub.loc[i, TARGETS] = p
        except Exception as e:
            n_failed += 1
            print(f"  study {i} fell back to 0.5: {type(e).__name__}: {e}")
    infer_seconds = time.time() - t0

path = write_submission(sub)
per_study = infer_seconds / max(len(sub), 1)
results.update({"infer_seconds_total": round(infer_seconds, 1),
                "infer_seconds_per_study": round(per_study, 2),
                "studies_failed_to_0.5": n_failed})

print(f"\nsubmission written: {path}  ({len(sub)} rows, {n_failed} fell back to 0.5)")
if per_study > 0:
    print(f"inference: {per_study:.2f} s/study (cold cache)")
    for n in (500, 1000, 2000, 5000):
        h = per_study * n / 3600
        flag = "OK" if h <= WORKING_LIMIT_H else ("tight" if h <= KAGGLE_LIMIT_H else "OVER 9h CAP")
        print(f"   {n:>5} hidden studies -> {h:5.2f} h   {flag}")
    print("   (hidden test size is unpublished — rules.md [verify]. Read the row you believe.)")

In [ ]:
# ---- final rules self-check: the submission contract (rules.md hard rules 3, 4, 10) -------------
def test_submission_contract(path: Path):
    out = pd.read_csv(path)
    assert path.name == "submission.csv", f"wrong filename: {path.name}"
    assert list(out.columns) == ["StudyInstanceUID"] + TARGETS, f"wrong columns: {list(out.columns)}"
    assert len(out) == len(test), f"{len(out)} rows for {len(test)} test studies"
    assert set(out["StudyInstanceUID"].astype(str)) == set(test["StudyInstanceUID"].astype(str)), "UID mismatch"
    v = out[TARGETS].values
    assert np.isfinite(v).all(), "NaN or inf in the submission"
    assert (v >= 0).all() and (v <= 1).all(), "probabilities outside [0, 1]"
    assert not out["StudyInstanceUID"].duplicated().any(), "duplicate study rows"
    print(f"submission contract: 7/7 assertions pass ({len(out)} rows)")

test_submission_contract(path)

results["total_runtime_hours"] = round(elapsed_h(), 3)
with open(WORK / f"{EXPERIMENT_ID}_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))
budget_check("end of notebook")

## 9. What to copy into `experiments.md`

From `baseline-v1_results.json`, fill the **Baseline lineages** table (lineage `L0`):

- `baseline_cv` → baseline CV, with `seeds`, `n_folds`, `eval_repeats` as the protocol
- `sigma` and `significance_threshold_2sigma` → **the bar every later experiment must clear**
- `dropped_cells / total_cells` → comparability key
- `labeller_gold_auc` → what the reports alone are worth; the image model must beat it
- `infer_seconds_per_study` → the Efficiency-track figure
- `train_gpu_seconds` → charge against the weekly GPU ledger (`AGENTS.md` §8)

Then submit the notebook once to fill the public-LB cell. Per `AGENTS.md` §13.1 the baseline moves
only on ≥20% error-gap closure or a genuinely new approach family — everything else is an accepted
improvement inside `L0`.

**Known weaknesses of this baseline, in priority order** — these are the first experiments, not
defects to hide:
1. The labeller is crude keyword matching; every image model inherits its errors. Highest leverage.
2. `resnet18` at 192 px on 12 windows is a smoke-test capacity, chosen to fit the GPU budget.
3. Three planes at four windows each ignores most of the volume.
4. Pretrained weights need an attached dataset; a random-init run is not a valid baseline.